# Review-1: Data Preprocessing, Feature Engineering & EDA
## Faithfulness-Aware Explainability Audit of Multilingual Transformer Models for Code-Mixed Hate Speech Detection

**Academic Integrity Statement**: All analysis in this notebook is performed strictly on raw dataset files without data fabrication, cherry-picking, or label alteration. Source files inside `datasets/` remain 100% untouched.

### 1. Project Overview
This project investigates code-mixed hate speech detection and explainability auditing across three language pairs:
1. **Hindi-English** (HECM)
2. **Tamil-English** (DravidianCodeMix)
3. **Malayalam-English** (DravidianCodeMix)

This Review-1 notebook covers **Data Loading, Data Quality Audits, Text Preprocessing, Feature Extraction (13 features), and Exploratory Data Analysis**.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set UTF-8 encoding & working directory
sys.path.append('..')
from src.preprocessing import verify_dataset_hashes, load_hindi, load_tamil, load_malayalam, load_all_datasets, compute_duplicate_audit, preprocess_text
from src.feature_engineering import extract_features, get_feature_definitions

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid')
print('Setup complete!')

### 2. Dataset Overview & Integrity Verification
We compute SHA-256 hashes of all raw dataset files before proceeding.

In [ ]:
initial_hashes = verify_dataset_hashes('../datasets')
for fname, h in initial_hashes.items():
    print(f'{fname}: {h}')

### 3. Dataset Loading & HECM Overview
We load all datasets cleanly. HECM text and class labels are loaded and aligned.

In [ ]:
df_hindi = load_hindi('../datasets/Hindi')
df_tamil = load_tamil('../datasets/TAMIL')
df_malayalam = load_malayalam('../datasets/MALAYALAM')

print(f'HECM Hindi shape: {df_hindi.shape}')
print(f'Tamil shape: {df_tamil.shape}')
print(f'Malayalam shape: {df_malayalam.shape}')


### 4. Data Quality Checks
We audit missing values, exact duplicate rows, duplicate text instances, and conflicting text instances.

In [ ]:
audit_h = compute_duplicate_audit(df_hindi)
audit_t = compute_duplicate_audit(df_tamil)
audit_m = compute_duplicate_audit(df_malayalam)

df_quality = pd.DataFrame([
    {'Dataset': 'HECM Hindi-English', **audit_h},
    {'Dataset': 'Tamil-English', **audit_t},
    {'Dataset': 'Malayalam-English', **audit_m}
])
display(df_quality)

### 5. Missing Values
Verifying NA counts across all raw fields.

In [ ]:
print('Hindi missing:', df_hindi.isna().sum().to_dict())
print('Tamil missing:', df_tamil.isna().sum().to_dict())
print('Malayalam missing:', df_malayalam.isna().sum().to_dict())

### 6. Duplicate Analysis
Analysis of text duplicates and conflicting labels (same text, different ground-truth label).

In [ ]:
print('Tamil Conflicting Texts (sample):')
t_groups = df_tamil.groupby('text')['label'].nunique()
conf_t_texts = t_groups[t_groups > 1].index
display(df_tamil[df_tamil['text'].isin(conf_t_texts[:3])].head(6))

### 7. Label Distribution
Plotting class distributions separately per language to preserve independent label systems.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
df_hindi['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#2b5c8f', '#d95f02'])
axes[0].set_title('Hindi Class Distribution (Class 0 vs Class 1)')

df_tamil[df_tamil['split']=='train']['label'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='#7570b3')
axes[1].set_title('Tamil Train Class Distribution (0-5)')

df_malayalam[df_malayalam['split']=='train']['label'].value_counts().plot(kind='bar', ax=axes[2], color='#1b9e77')
axes[2].set_title('Malayalam Train Class Distribution')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### 8. Text Length Analysis
Preprocessing text and extracting 13 linguistic features.

In [ ]:
df_all = load_all_datasets('../datasets')
df_all['clean_text'] = df_all['text'].apply(preprocess_text)
df_all_feats = extract_features(df_all, text_col='clean_text', lang_col='language')
print('Features extracted shape:', df_all_feats.shape)

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df_all_feats, x='character_count', hue='language', fill=True, common_norm=False)
plt.title('Text Character Length Distribution across Language Pairs')
plt.xlim(0, 400)
plt.show()

### 9. Word Count Analysis

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df_all_feats, x='word_count', hue='language', fill=True, common_norm=False)
plt.title('Word Count Distribution across Language Pairs')
plt.xlim(0, 80)
plt.show()

### 10. Script Analysis
Comparing counts of Latin/English script characters vs Native Indic script characters.

In [ ]:
display(df_all_feats.groupby('language')[['english_character_count', 'native_script_character_count']].mean())

### 11. English / Native Script Ratio

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df_all_feats, x='language', y='native_script_ratio', hue='language', legend=False)
plt.title('Native Script Ratio Distribution')
plt.show()

### 12. Code-Mixing Analysis
**Script-Based Code-Mixing Ratio Definition**:
$$ \text{script_code_mixing_ratio} = \frac{\min(\text{English_Chars}, \text{Native_Chars})}{\text{English_Chars} + \text{Native_Chars}} $$

**Documented Limitation**: This metric measures code-mixing based on script differences. Regional text written in Roman/Latin script (transliterated text) is counted as English script, resulting in an underestimation of code-mixing.

In [ ]:
plt.figure(figsize=(9, 5))
sns.violinplot(data=df_all_feats, x='language', y='script_code_mixing_ratio', hue='language', inner='quartile', legend=False)
plt.title('Script-Based Code-Mixing Ratio Distribution')
plt.show()

### 13. Language-Switch Analysis
Counting transitions between English script and native script.

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df_all_feats, x='language', y='language_switch_count', hue='language', legend=False)
plt.title('Language Switch Count Distribution')
plt.ylim(-0.5, 20)
plt.show()

### 14. Class vs Code-Mixing Analysis
Analyzed separately per language to prevent merging incompatible label systems.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
df_h = df_all_feats[df_all_feats['language']=='Hindi-English']
df_t = df_all_feats[df_all_feats['language']=='Tamil-English']
df_m = df_all_feats[df_all_feats['language']=='Malayalam-English']

sns.boxplot(data=df_h, x='label', y='script_code_mixing_ratio', ax=axes[0], hue='label', legend=False)
axes[0].set_title('Hindi-English (Class 0 vs Class 1)')

sns.boxplot(data=df_t, x='label', y='script_code_mixing_ratio', ax=axes[1], hue='label', legend=False)
axes[1].set_title('Tamil-English (0-5)')

sns.boxplot(data=df_m, x='label', y='script_code_mixing_ratio', ax=axes[2], hue='label', legend=False)
axes[2].set_title('Malayalam-English')
axes[2].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

### 15. Cross-Language Comparison

In [ ]:
avg_summary = df_all_feats.groupby('language')[['character_count', 'word_count', 'native_script_ratio', 'script_code_mixing_ratio', 'language_switch_count']].mean()
display(avg_summary)

### 16. Key Findings
1. **Zero Data Removal**: Retained 100% of 73,286 dataset rows across all splits.
2. **HECM Dataset Characteristics**: HECM text standardizes URLs and handles while preserving code-mixed Hindi text.
3. **Class Imbalance**: Malayalam (88.35% Non-offensive) and Tamil (72.42% Class 0) show heavy class imbalance, whereas HECM Hindi has 69.83% Class 1 vs 30.17% Class 0.
4. **Script-Based Code-Mixing**: Average script-based code-mixing ratio is highest in Hindi-English (0.1068), followed by Malayalam-English (0.0528) and Tamil-English (0.0470).

### 17. Limitations
1. **Script-Based Detection Limitation**: Transliterated text (e.g. hinglish/tanglish/manglish in Latin script) is identified as English script, underestimating actual linguistic code-mixing.
2. **Label Incompatibility**: Each language dataset has a distinct label system, prohibiting cross-lingual label merging.
3. **Dataset Noise**: Duplicate text instances with conflicting labels exist in Tamil (15) and Malayalam (5) raw datasets.